# SHUNO · Motor GPU en Google Colab T4

Este notebook levanta **ACE-Step 1.5** como API para que la web de SHUNO (Vercel) pueda generar canciones.

Flujo: **SHUNO Web → Vercel API → túnel HTTPS → ACE-Step en Colab GPU**.


In [ ]:
# 1) Verificar GPU
!nvidia-smi


In [ ]:
# 2) Instalar herramientas y ACE-Step 1.5
!apt-get -qq update
!apt-get -qq install -y ffmpeg git curl
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!rm -rf /content/ACE-Step-1.5
!git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git /content/ACE-Step-1.5
%cd /content/ACE-Step-1.5
!uv sync


In [ ]:
# 3) Crear una clave privada para este arranque
import os, secrets
SHUNO_API_KEY = 'shuno_' + secrets.token_urlsafe(24)
os.environ['ACESTEP_API_KEY'] = SHUNO_API_KEY
os.environ['ACESTEP_INIT_LLM'] = 'true'
print('API KEY creada. Guardala para configurar Vercel al final.')


In [ ]:
# 4) Levantar ACE-Step API en segundo plano
import subprocess, time, os, pathlib
log = open('/content/shuno_acestep.log', 'w')
proc = subprocess.Popen([
    'uv','run','acestep-api',
    '--host','0.0.0.0',
    '--port','8001',
    '--api-key', SHUNO_API_KEY,
    '--lm-model-path','acestep-5Hz-lm-0.6B'
], cwd='/content/ACE-Step-1.5', stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
print('PID:', proc.pid)
print('Esperando que arranque el servidor y descargue/cargue modelos...')
time.sleep(20)
!tail -n 30 /content/shuno_acestep.log


In [ ]:
# 5) Comprobar salud local
import requests, time
for i in range(60):
    try:
        r = requests.get('http://127.0.0.1:8001/health', timeout=5)
        print('Health:', r.status_code, r.text[:500])
        if r.ok: break
    except Exception as e:
        if i % 5 == 0: print('Aún iniciando...', e)
    time.sleep(5)


In [ ]:
# 6) Instalar Cloudflare Tunnel (URL pública temporal)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, time, re, os
tunnel_log = open('/content/shuno_tunnel.log','w')
tunnel = subprocess.Popen([
    'cloudflared','tunnel','--url','http://127.0.0.1:8001','--no-autoupdate'
], stdout=tunnel_log, stderr=subprocess.STDOUT)
print('Tunnel PID:', tunnel.pid)
time.sleep(8)
text = open('/content/shuno_tunnel.log', errors='ignore').read()
match = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', text)
SHUNO_ENGINE_URL = match.group(0) if match else None
print('SHUNO_ENGINE_URL =', SHUNO_ENGINE_URL)
print('SHUNO_ENGINE_API_KEY =', SHUNO_API_KEY)


## 7) Conectar Vercel

En el proyecto **shuno** de Vercel agregá estas variables:

- `SHUNO_ENGINE_URL` = la URL `https://....trycloudflare.com` impresa arriba
- `SHUNO_ENGINE_API_KEY` = la clave `shuno_...` impresa arriba

Después hacé un redeploy de SHUNO. La URL de Cloudflare es temporal y cambia cada vez que reiniciás Colab.


In [ ]:
# 8) Prueba directa opcional desde Colab
import requests, json, time
headers={'Authorization': f'Bearer {SHUNO_API_KEY}', 'Content-Type':'application/json'}
payload={
  'prompt':'zamba argentina romántica, guitarra criolla, bombo legüero sutil, voz masculina emotiva',
  'lyrics':'[Verse 1]\\nEsta es una prueba de SHUNO\\n\\n[Chorus]\\nQuiero cantar',
  'audio_duration':60,
  'bpm':84,
  'key_scale':'G Major',
  'time_signature':'3',
  'batch_size':1,
  'audio_format':'wav',
  'thinking':True,
  'use_format':True,
  'task_type':'text2music'
}
# Descomentá para probar:
# r=requests.post('http://127.0.0.1:8001/release_task',headers=headers,json=payload,timeout=60)
# print(r.status_code, r.text)
